# MirrorTopology Step 1 — Phase B・B-3-0 smoke（v0.3）
**外側の launcher lock**（cell 0：検証対象 commit C・期待 inventory SHA）→ fresh scratch checkout（clean tree）→ inventory が科学 pins／script／test を束縛（一方向：pins は自分の commit や inventory を含まない）→ 環境 lock → smoke script（required gate 16 の exact inventory）→ 実資産 pytest（**期待 node 集合の完全一致**・SKIP/DESELECT 0）→ 最終 record（全段階の合成；自身を inventory に含めない）。attempt ごとに新規 run directory。label は解放しない。

In [1]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell). These values are NOT inside the verification-target tree.
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '27c3b022f905fff16365c814f5ab5db421ba50c0'
EXPECTED_INVENTORY_SHA256 = '13c7f89a42b7cfeb99c95dadf5a3a110d564133e15d0ba7e3fa272e95f249449'
LAUNCHER_ID = 'MirrorTopology_Step1_B3_0_smoke_v0.3'


In [2]:
# --- 1. fresh scratch checkout at C; clean tree; inventory bound; science pins / script / required test bound through the inventory (one direction)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256), 'launcher lock not filled'
RUN=f'/content/b3_0_runs/{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'{RUN}/scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/b3/b3_0_pins.json'
assert sha(PINS)==inv['b3_sha256']['b3/b3_0_pins.json'] and sha(f'{PHASEB}/b3/b3_0_smoke.py')==inv['b3_sha256']['b3/b3_0_smoke.py'] and sha(f'{PHASEB}/tests/test_b2_tranche23.py')==inv['tests_sha256']['test_b2_tranche23.py']
pins=json.load(open(PINS)); assert 'repo' not in pins and pins['engine_version']==inv['engine_version']
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], run_dir=RUN); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


{'launcher_id': 'MirrorTopology_Step1_B3_0_smoke_v0.3', 'repo_url': 'https://github.com/tsujikeita/mirror-topology.git', 'commit': '27c3b022f905fff16365c814f5ab5db421ba50c0', 'inventory_sha256': '13c7f89a42b7cfeb99c95dadf5a3a110d564133e15d0ba7e3fa272e95f249449', 'pins_sha256': '65bb7d25ddb89328560076bae795427b2fc33cd0dd960af4e0a09235d31b1c99', 'engine_version': '0.43.0', 'run_dir': '/content/b3_0_runs/20260917T042949Z'}


In [3]:
# --- 2. environment lock (installed before any computation; the script re-measures live versions and BLAS pools as a required gate)
ex=pins['environment']
subprocess.run(['pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",f"camb=={ex['camb']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'; os.environ.pop('PYTEST_ADDOPTS', None)
import platform, numpy, scipy, healpy, ot, camb
live=dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, healpy=healpy.__version__, pot=ot.__version__, camb=camb.__version__)
mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed: {mism}'; print('environment lock OK', live)


environment lock OK {'python': '3.13.15', 'numpy': '2.1.3', 'scipy': '1.16.3', 'healpy': '1.20.0', 'pot': '0.9.7.post1', 'camb': '2.0.4'}


In [4]:
# --- 3. B-3-0 smoke script (fresh OUT/smoke; inventory-bound pins; required gates; evidence + run manifest)
SM=f'{OUT}/smoke'
rc=subprocess.run([sys.executable,f'{PHASEB}/b3/b3_0_smoke.py','--mt',MT,'--phaseb',PHASEB,'--out',SM,'--profile','smoke'],capture_output=True,text=True)
open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-1500:])
rm=json.load(open(f'{SM}/b3_0_run_manifest.json')); print('script B3_0_PASS:', rm['B3_0_PASS'], rm['failures'])


B3_0_PASS = True | stage: complete | gates: {"G_pins_loaded": true, "G_engine_inventory": true, "G_script_sha": true, "G_env_lock": true, "G_assets_sha": true, "G_cov_manifest_binding": true, "G_cov_array_sha": true, "G_sqrt_hard": true, "G_roles_shared_latent": true, "G_finite_banks": true, "G_iso_engine_matches_A5_null": true, "G_T10_stratified_equals_literal": true, "G_T10_prefix_extension_contrast": true, "G_evaluation_ok": true, "G_checkpoint_roundtrip": true, "G_evidence_saved": true}
run manifest written; B3_0_PASS = True

script B3_0_PASS: True []


In [5]:
# --- 4. required real-asset pytest: exact expected node set (collect-only from the pinned tree, must include the 3 legacy nodes), no env filters, JUnit
env=dict(os.environ, PHASEB_ROOT=PHASEB, B3_MT=MT, B3_A10_NB=f"{MT}/{pins['a10_frozen']['notebook']}", B3_A10_CHECKPOINT=f'{PHASEB}/tests/reference_assets/a10a_calibration_official.npz'); env.pop('PYTEST_ADDOPTS', None)
files=[f'{PHASEB}/tests/test_b2_tranche23.py',f'{PHASEB}/tests/test_b1.py',f'{PHASEB}/tests/test_b1_contracts.py']
col=subprocess.run([sys.executable,'-m','pytest','--rootdir',PHASEB,'--collect-only','-q','-p','no:cacheprovider',*files],capture_output=True,text=True,cwd=PHASEB,env=env)
open(f'{OUT}/pytest_collection_stdout.txt','w').write(col.stdout); open(f'{OUT}/pytest_collection_stderr.txt','w').write(col.stderr)
if col.returncode != 0: raise RuntimeError(f'pytest collection failed: {col.returncode}')
expected={l.strip() for l in col.stdout.splitlines() if '::' in l}
REQ_LEGACY={'tests/test_b2_tranche23.py::test_legacy_kernel_assets_and_representation_gates','tests/test_b2_tranche23.py::test_legacy_kernel_bit_identical_to_frozen_notebook_cell','tests/test_b2_tranche23.py::test_legacy_kernel_cross_environment_agreement_with_a10_official'}
assert REQ_LEGACY <= expected, 'required legacy nodes not collected'
rp=subprocess.run([sys.executable,'-m','pytest','--rootdir',PHASEB,'-v','-p','no:cacheprovider','--junitxml',f'{OUT}/pytest_b3_0.xml',*files],capture_output=True,text=True,cwd=PHASEB,env=env)
open(f'{OUT}/pytest_b3_0_stdout.txt','w').write(rp.stdout); open(f'{OUT}/pytest_b3_0_stderr.txt','w').write(rp.stderr)
import xml.etree.ElementTree as ET; root=ET.parse(f'{OUT}/pytest_b3_0.xml').getroot(); cases=[c for c in root.iter('testcase')]
def node_of(c):
    cls=c.attrib['classname']; f=cls.replace('.','/')+'.py'; return f"{f}::{c.attrib['name']}"
ran={node_of(c) for c in cases}; ok={node_of(c) for c in cases if not any(c.find(tag) is not None for tag in ('failure','error','skipped'))}  # properties/system-out/system-err are not failures
pytest_ok=(col.returncode==0 and rp.returncode==0 and len(cases)==len(expected) and ran==expected and ok==expected and REQ_LEGACY<=ok)
print(rp.stdout[-600:]); print('expected', len(expected), 'ran', len(ran), 'passed', len(ok), 'pytest_ok', pytest_ok)


_ _ _ _ _ _ 

    def get_kernelapp():
>     return get_ipython().kernel.parent
             ^^^^^^^^^^^^^^^^^^^^
E     AttributeError: 'NoneType' object has no attribute 'kernel'

/usr/local/lib/python3.13/dist-packages/google/colab/_ipython.py:28: AttributeError
- generated xml file: /content/b3_0_runs/20260917T042949Z/out/pytest_b3_0.xml --
=========================== short test summary info ============================
FAILED tests/test_b2_tranche23.py::test_legacy_kernel_bit_identical_to_frozen_notebook_cell
======================== 1 failed, 38 passed in 10.51s =========================

expected 39 ran 39 passed 38 pytest_ok False


In [6]:
# --- 5. final record composed from all stages (the record itself is excluded from the hashed inventory; an outer return list is written separately)
def inventory(root, exclude=()):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue
            out[rel]=dict(sha256=sha(p), bytes=os.path.getsize(p))
    return out
# A stale PASS manifest must not override failure of the current subprocess.
script_ok = bool(rc.returncode == 0 and rm.get('B3_0_PASS') is True)
final=dict(launcher=lock, B3_0_PASS=bool(script_ok and pytest_ok), stages=dict(checkout=True, environment=live, script_returncode=rc.returncode, script_pass=script_ok, stored_script_pass=rm.get('B3_0_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures',[]), pytest=dict(collection_returncode=col.returncode, returncode=rp.returncode, expected=sorted(expected), passed=sorted(ok), required_legacy_present=REQ_LEGACY<=ok, ok=pytest_ok)), execution_profile=rm.get('execution_profile'))
final['output_inventory']=inventory(OUT, exclude=('b3_0_final_record.json','b3_0_return_list.json'))
json.dump(final, open(f'{OUT}/b3_0_final_record.json','w'), indent=1); json.dump(dict(final_record_sha256=sha(f'{OUT}/b3_0_final_record.json'), files=final['output_inventory']), open(f'{OUT}/b3_0_return_list.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('B3_0_PASS',)}, indent=1), 'run dir:', RUN)


{
 "B3_0_PASS": false
} run dir: /content/b3_0_runs/20260917T042949Z


## 監査へ渡すもの
`RUN/out/` 全体（launcher_lock・smoke/ の run manifest と証拠・pytest XML／log・final record・return list）。前 attempt の run directory は削除しない。